In [ ]:
!pip install -q librosa scikit-learn
import librosa
import pandas as pd
import seaborn as sns
import numpy as np
import os

In [ ]:
!rm -rf DrumRecognizer
!git clone https://github.com/DongHaShin03/DrumRecognizer.git

Cloning into 'DrumRecognizer'...
remote: Enumerating objects: 197, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 197 (delta 5), reused 43 (delta 0), pack-reused 142 (from 1)
Receiving objects: 100% (197/197), 66.90 MiB | 4.72 MiB/s, done.
Resolving deltas: 100% (12/12), done.


In [ ]:
paths = {
    'kick': '/content/DrumRecognizer/drum_set/kick/',
    'snare': '/content/DrumRecognizer/drum_set/snare/',
    'tom': '/content/DrumRecognizer/drum_set/toms/',
    'overhead': '/content/DrumRecognizer/drum_set/overheads/'
}

n_mfcc=10

In [ ]:
def extract_features(file_path, label, sr=22050, n_mfcc=n_mfcc):
  y, sr = librosa.load(file_path, sr=22050, mono=True)
  mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
  zcr = librosa.feature.zero_crossing_rate(y)
  cent = librosa.feature.spectral_centroid(y=y, sr=sr)

  row = {
        'label': label,
        'zcr': float(np.mean(zcr)),
        'cent': float(np.mean(cent)),
    }
  mfcc_means = np.mean(mfcc, axis=1)
  for i, value in enumerate(mfcc_means, start=1):
    row[f'mfcc_{i}'] = float(value)

  return row


In [ ]:
rows=[]
for label, folder_name in paths.items():
  for file in os.listdir(folder_name):
    directory = f"{paths[label]}" + f"{file}"
    if file.endswith(".wav"):
      rows.append(extract_features(directory, label))

df = pd.DataFrame(rows)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
output_path = "/content/drive/MyDrive/2 - Side Projects/DrumRecognizer/imgs/drum_features.csv"

df.to_csv(output_path, index=False)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
